<a href="https://colab.research.google.com/github/CamilyVoltan/chatbot_ClariceLispector/blob/main/Trabalho_de_PLN_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TRABALHO DE LINGUAGEM DE PROCESSAMENTO NATURAL**

Escolher um autor e monte um modelo de PLN que seja tal qual o autor e  seja via chat. Utilizando modelo transformers e anagramas.

In [ ]:
!pip install transformers torch datasets accelerate
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Limpeza e preparação do Corpus**

In [ ]:
import re
import unicodedata

entrada = "corpus_clarice_lispector.txt"
saida = "corpus_clarice.txt"

with open(entrada, "r", encoding="utf-8", errors="ignore") as f:
    texto = f.read()

texto = re.sub(r'[^\x00-\x7F]+', ' ', texto)
texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
texto = re.sub(r'([a-zA-Z])\s+([a-zA-Z])', r'\1\2', texto)
texto = re.sub(r'LA OS DE FAM LIA.*?A Autora', ' ', texto, flags=re.DOTALL | re.IGNORECASE)
texto = re.sub(r'Sobre a obra:.*?O triunfo', ' ', texto, flags=re.DOTALL | re.IGNORECASE)
texto = re.sub(r'\s-\s', ' ', texto)
texto = re.sub(r'\f', ' ', texto)
texto = re.sub(r'LeLivros.*?[\n\r]', ' ', texto, flags=re.IGNORECASE)
texto = re.sub(r'SUMÁRIO.*?(?=\n[A-ZÁÉÍÓÚ])', ' ', texto, flags=re.DOTALL)
texto = re.sub(r'DADOS\s+DE\s+COPYRIGHT.*?(?=\n[A-ZÁÉÍÓÚ])', ' ', texto, flags=re.DOTALL)

texto = re.sub(r'\s+', ' ', texto)

texto = re.sub(r'http\S+|www\.\S+', '', texto)
texto = re.sub(r'[^\x00-\x7F]+', ' ', texto)

texto = re.sub(r'\b(SUMÁRIO|Prefácio|Organização|Clarice Lispector)\b', '', texto, flags=re.IGNORECASE)

texto = texto.strip()
texto = re.sub(r'\s{2,}', ' ', texto)

with open(saida, "w", encoding="utf-8") as f:
    f.write(texto)

print("✅ Limpeza concluída!")
print(f"Arquivo limpo salvo como: {saida}")

# Mostra as 500 primeiras palavras para conferir o resultado
print("\nPré-visualização do texto limpo:\n")
print(" ".join(texto.split()[:500]))

✅ Limpeza concluída!
Arquivo limpo salvo como: corpus_clarice.txt

Pré-visualização do texto limpo:

DADOSDECOPYRIGHTSobrea obra: Apresenteobradisponibilizadapelaequipe expressamenteproibidae totalmenterepudivela venda, aluguel, ouquaisquerusocomercialdopresentecontedoSobren s: O todae qualquerpessoa. Vocpodeencontrarmaisobrasemnossosite: "Quandoo mundoestiverunidonabuscadoconhecimento, en omaislutandopordinheiroe poder, ento nossasociedadepoderenfimevoluira umnovon vel." ClariceLispectorTODOSOSCONTOSPrefcioe organizao deBENJAMINMOSERSUMRIOParapularo Sumrio, cliqueaqui. Glamoure gramticaPRIMEIRASHISTRIASO triunfoObsesso OdelrioEue JimmyHistriainterrompidaA fugaTrechoCartasa HermengardoGertrudespedeumconselhoMaisdoisb bedosLAOSDEFAMLIADevaneioe embriaguezdumaraparigaAmorUmagalinhaA imitao darosaFelizaniversrioA menormulherdomundoO jantarPreciosidadeOslaosdefamliaComeosdeumafortunaMistrioemS oCristv oO crimedoprofessordematemticaO bfaloA LEGIO ESTRANGEIRAOsdesastresdeSofiaA repartio dosp

In [ ]:
!head -n 20 corpus_clarice.txt

DADOSDECOPYRIGHTSobrea obra: Apresenteobradisponibilizadapelaequipe expressamenteproibidae totalmenterepudivela venda, aluguel, ouquaisquerusocomercialdopresentecontedoSobren s: O todae qualquerpessoa. Vocpodeencontrarmaisobrasemnossosite: "Quandoo mundoestiverunidonabuscadoconhecimento, en omaislutandopordinheiroe poder, ento nossasociedadepoderenfimevoluira umnovon vel." ClariceLispectorTODOSOSCONTOSPrefcioe organizao deBENJAMINMOSERSUMRIOParapularo Sumrio, cliqueaqui. Glamoure gramticaPRIMEIRASHISTRIASO triunfoObsesso OdelrioEue JimmyHistriainterrompidaA fugaTrechoCartasa HermengardoGertrudespedeumconselhoMaisdoisb bedosLAOSDEFAMLIADevaneioe embriaguezdumaraparigaAmorUmagalinhaA imitao darosaFelizaniversrioA menormulherdomundoO jantarPreciosidadeOslaosdefamliaComeosdeumafortunaMistrioemS oCristv oO crimedoprofessordematemticaO bfaloA LEGIO ESTRANGEIRAOsdesastresdeSofiaA repartio dosp esA mensagemMacacosO ovoe agalinhaTentao Viagema PetrpolisA soluo Evoluo deumamiopiaA quintahistriaU

**Preparação do Dataset**

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [ ]:
model_name = "pierreguillou/gpt2-small-portuguese"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)


In [ ]:
from transformers import TextDataset, DataCollatorForLanguageModeling

def load_dataset(file_path, tokenizer, block_size=128):
    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size
    )
    return dataset

dataset = load_dataset("corpus_clarice.txt", tokenizer)

/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [ ]:
# CÉLULA 1
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import TextDataset, DataCollatorForLanguageModeling

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# CÉLULA 2
def load_dataset(file_path, tokenizer, block_size=128):
    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size
    )
    return dataset

dataset = load_dataset("corpus_clarice.txt", tokenizer)


# CÉLULA 3
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./clarice_clm_model",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
    logging_steps=500,
)


# CÉLULA 4
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

# Comando para iniciar o treinamento:
# trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (327374 > 1024). Running this sequence through the model will result in indexing errors


**Treinamento do Modelo**

In [ ]:
from transformers import DataCollatorForLanguageModeling

# Seu código onde o erro apareceu:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./clarice_model",        # pasta onde o modelo será salvo
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    prediction_loss_only=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:


Abort: 

In [ ]:
model.save_pretrained("./clarice_final")
tokenizer.save_pretrained("./clarice_final")

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="./clarice_final", tokenizer="./clarice_final")

prompt = "Escreva uma reflexão sobre o amor:"
resposta = generator(prompt, max_length=120, num_return_sequences=1)

print(resposta[0]["generated_text"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math
import re
import pandas as pd

# --- CONFIGURAÇÃO ---
modelo_base = "pierreguillou/gpt2-small-portuguese"
corpus = "corpus_clarice.txt"

# CORREÇÃO DE TIPO: A extensão é .csv (Comma-Separated Values), não .cvs
output_csv = "../resultado_avaliacao.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

# --- CARREGAMENTO DO MODELO ---
# 1. Use a classe AutoTokenizer diretamente
# 2. Salve na variável "tokenizer", e não na variável "transformers"
tokenizer = AutoTokenizer.from_pretrained(modelo_base)

if tokenizer.pad_token is None:
     tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

# 3. Use a classe AutoModelForCausalLM diretamente
model = AutoModelForCausalLM.from_pretrained(modelo_base).to(device)
model.eval() # Coloca o modelo em modo de avaliação (importante!)

# --- PROCESSAMENTO DOS DADOS ---
with open(corpus, "r", encoding="utf-8") as f:
    val_texts = [t.strip() for t in f.readlines() if len(t.strip()) > 30]

inputs_list = [tokenizer(t, return_tensors="pt")["input_ids"].to(device) for t in val_texts]

# --- FUNÇÃO DE AVALIAÇÃO (Perplexity) ---
def compute_perplexity(input_list, max_samples=100):
    total_loss = 0
    total_token = 0

    # Limita o número de amostras para não demorar uma eternidade
    samples_to_process = input_list[:max_samples]
    if not samples_to_process:
        print("Aviso: 'inputs_list' está vazio. Não é possível calcular a perplexidade.")
        return 0

    for inputs in samples_to_process:
        if inputs.size(1) == 0: # Pula tensores vazios
            continue

        with torch.no_grad():
            outputs = model(inputs, labels=inputs)

        loss = outputs.loss.item()
        num_tokens = inputs.size(1)

        total_loss += loss * num_tokens
        total_token += num_tokens

    if total_token == 0:
        print("Aviso: Nenhum token processado. Não é possível calcular a perplexidade.")
        return 0

    perplexity = math.exp(total_loss / total_token)
    return perplexity

# CORREÇÃO DE VARIÁVEL: Você chamou a variável de "ppl" no DataFrame,
# mas a salvou como "pipline" (que também é um erro de digitação de "pipeline").
# Vamos usar "ppl" que é mais curto.
pipeline = compute_perplexity(inputs_list)
print(f"Perplexity Média: {ppl:.2f}")

# --- FUNÇÃO DE GERAÇÃO DE AMOSTRAS ---
def gerar_amostras(model, tokenizer, prompts, max_new_tokens=200):
    samples = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.9,
                top_p=0.92,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
            texto = tokenizer.decode(outputs[0], skip_special_tokens=True)
        samples.append(texto)
    return samples

def contar_palavras_sentencas(textos):
    metrics = []
    for t in textos:
        num_palavras = len(t.split())
        # Garante que sentenças vazias não sejam contadas
        num_sentencas = len([s for s in re.split(r'[.!?]', t) if len(s.strip()) > 0])
        metrics.append({"num_palavras": num_palavras, "num_sentencas": num_sentencas})
    return metrics

# --- EXECUÇÃO E SALVAMENTO ---
prompts = [
    "Ela acordou com a sensação de que o mundo se tornara um espelho quebrado.",
    "Dentro de mim, algo silencioso crescia como uma flor que não sabia o nome.",
    "A solidão tem um gosto que aprendi a não temer.",
    "O instante era mais do que tempo, era revelação.",
]

amostras = gerar_amostras(model, tokenizer, prompts)
metrics = contar_palavras_sentencas(amostras)

print("\n📝 Resultados da Avaliação:")
for i, t in enumerate(amostras):
    print(f"Prompt: {prompts[i]}")
    print(f"Texto gerado: {t}")
    print(f"Palavras: {metrics[i]['num_palavras']}, Sentenças: {metrics[i]['num_sentencas']}")
    print("-"*50)

# Criação do DataFrame
df = pd.DataFrame({
    "prompt": prompts,
    "texto_gerado": amostras,
    # Aqui usamos a variável "ppl" que calculamos
    "perplexity": [ppl] * len(amostras),
    "num_palavras": [m['num_palavras'] for m in metrics],
    "num_sentencas": [m['num_sentencas'] for m in metrics]
})

df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"\n✅ Relatório salvo em: {output_csv}")
#%%

Usando device: cpu


tokenizer_config.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

IndexError: index out of range in self